# Trabalho de Aprendizado de Maquina

Aqui descrevemos o tal do dataset assim como explicar a natureza do problema.

- Podemos fazer uma Regressao usando a previsao das notas

- Podemos tambem classificar em conceitos e fazer uma classificacao (Oq imagino ser mais interessante)



## 1. imports e load do dataset



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ferramentas de Validação Cruzada e Pré-processamento
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Algoritmos para Spot Checking (foco em Classificação)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC

# Carregando o dataset
file_path = 'Gaming_Academic_Performance_updated.csv'
df = pd.read_csv(file_path)

# Exibindo as primeiras linhas e informações básicas do dataset
display(df.head())
print("\nInformações do Dataset:")
df.info()

## 2. Analise dos dados

Aqui podemos ter as metricas estatisticas dos atributos numericos, assim como as classes dos atributos categoricos

In [ ]:
# Análise Preliminar de Dados Numéricos
print("--- Estatísticas Descritivas (Dados Numéricos) ---")
display(df.describe())

# Análise Preliminar de Dados Categóricos
print("\n--- Valores e Frequências (Dados Categóricos) ---")
categorical_cols = df.select_dtypes(exclude='number').columns

for col in categorical_cols:
    print(f"\nValores para a coluna '{col}':")
    display(df[col].value_counts())

# Verificando valores faltantes e instâncias duplicadas
print("\nValores faltantes por coluna:")
display(df.isna().sum())
print(f"Instâncias duplicadas (ignorando student_id): {df.drop(columns=['student_id']).duplicated().sum()}")

In [ ]:
# Distribuição dos atributos numéricos
numeric_cols = df.select_dtypes(include='number').columns.drop('student_id')
df[numeric_cols].hist(bins=30, figsize=(15, 10))
plt.tight_layout()
plt.show()

In [ ]:
# Distribuição das notas, com os limites dos conceitos (D < 60 <= C < 75 <= B < 90 <= A)
plt.figure(figsize=(10, 4))
sns.histplot(df['grades'], bins=50)
for limite in [60, 75, 90, 100]:
    plt.axvline(limite, color='red', linestyle='--')
plt.title('Distribuição de grades')
plt.show()

print(f"Notas acima de 100: {(df['grades'] > 100).sum()} (máximo = {df['grades'].max():.2f})")
print(f"Notas iguais a 0: {(df['grades'] == 0).sum()}")
print(f"addiction_score negativos: {(df['addiction_score'] < 0).sum()} (mínimo = {df['addiction_score'].min():.2f})")

In [ ]:
# Correlação entre os atributos numéricos
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Matriz de correlação')
plt.show()

### Problemas identificados na análise

- Nao ha valores faltantes nem instancias duplicadas.
- `student_id` eh apenas um identificador da linha, sem informacao para a previsao.
- 134 notas estao acima de 100 (maximo de ~118.6), o que eh impossivel para uma nota. Tambem ha uma nota igual a 0.
- 107 valores de `addiction_score` sao negativos (minimo de ~-4.51), o que nao faz sentido para um score de vicio.
- Os atributos estao em escalas bem diferentes (ex.: `reaction_time_ms` vai de ~183 a ~348, `social_activity` de 0 a 5). Isso afeta algoritmos baseados em distancia (KNN) e redes neurais, entao sera necessario normalizar.
- `study_hours`, `gaming_hours`, `reaction_time_ms`, `addiction_score` e `device_usage` sao os atributos mais correlacionados com `grades`.

## 3. Limpeza dos dados

- Removemos `student_id`, que nao tem valor preditivo.
- Limitamos `grades` ao intervalo [0, 100], ja que notas acima de 100 nao existem. Assim mantemos as 8000 instancias e corrigimos os valores impossiveis.
- Limitamos `addiction_score` a valores >= 0 pelo mesmo motivo.

In [ ]:
df = df.drop(columns=['student_id'])

df['grades'] = df['grades'].clip(0, 100)
df['addiction_score'] = df['addiction_score'].clip(lower=0)

print(f"grades: min = {df['grades'].min():.2f}, max = {df['grades'].max():.2f}")
print(f"addiction_score: min = {df['addiction_score'].min():.2f}")

## 4. Definicao do atributo alvo

Transformamos a nota em conceitos, seguindo a escala da UFRGS:

- A: nota >= 90
- B: 75 <= nota < 90
- C: 60 <= nota < 75
- D: nota < 60

Depois disso a coluna `grades` precisa ser removida dos atributos de entrada, pois o conceito eh calculado diretamente a partir dela (vazamento do alvo).

In [ ]:
df['conceito'] = pd.cut(df['grades'], bins=[-np.inf, 60, 75, 90, np.inf], right=False, labels=['D', 'C', 'B', 'A'])

print("--- Distribuição dos conceitos ---")
display(df['conceito'].value_counts(normalize=True).sort_index())

df['conceito'].value_counts().sort_index().plot(kind='bar')
plt.title('Quantidade de alunos por conceito')
plt.show()

As classes estao levemente desbalanceadas (D ~39%, A ~17%), mas nao o suficiente para precisar de balanceamento. Para lidar com isso vamos usar validacao cruzada estratificada e metricas com media macro (ex.: F1 macro).

## 5. Transformacao dos atributos categoricos

Os campos `gender` e `gaming_genre` sao categoricos sem nocao de ordem, entao usamos o One-Hot Encoding para poder usar isso no treinamento

O campo de `stress_level` eh categorico com nocao de ordem, entao apenas substituimos Low, Medium e High por 1, 2 e 3.

In [ ]:
# 1. Ordinal Encoding para 'stress_level'
stress_mapping = {'Low': 1, 'Medium': 2, 'High': 3}
df['stress_level'] = df['stress_level'].map(stress_mapping)

# 2. One-Hot Encoding para 'gender' e 'gaming_genre'
# Utilizamos drop_first=False para manter todas as colunas de categorias (Female, Casual, etc.)
df = pd.get_dummies(df, columns=['gender', 'gaming_genre'], drop_first=False, dtype=int)

# Vamos ver como o dataset ficou após as transformações
print("--- Dataset após transformações (Primeiras 5 linhas) ---")
display(df.head())

print("\n--- Informações dos Tipos de Dados ---")
display(df.dtypes)

## 6. Separacao entre atributos (X) e alvo (y)

A normalizacao (`StandardScaler`) **nao** eh aplicada aqui. Ela sera colocada dentro de um `Pipeline` junto com cada modelo, para que a media e o desvio padrao sejam calculados apenas com os dados de treino de cada fold da validacao cruzada. Normalizar o dataset inteiro antes vazaria informacao dos dados de teste para o treino.

In [ ]:
X = df.drop(columns=['grades', 'conceito'])
y = df['conceito']

print(f"X: {X.shape}, y: {y.shape}")
display(X.head())

In [ ]:
# Salvando o dataset limpo e transformado (atributos + conceito)
df_limpo = X.copy()
df_limpo['conceito'] = y
df_limpo.to_csv('Gaming_Academic_Performance_limpo.csv', index=False)
print(f"Dataset salvo: {df_limpo.shape}")

## 7. Protocolo de avaliacao

- **Validacao cruzada estratificada repetida**: 5 folds x 10 repeticoes = 50 avaliacoes por modelo. A estratificacao mantem a proporcao dos conceitos em cada fold, e as repeticoes permitem analisar a media e a variabilidade do desempenho.
- O mesmo objeto `cv` (com seed fixa) eh usado para todos os modelos, entao todos sao avaliados exatamente nas mesmas divisoes de treino/teste.
- **Metrica principal: F1 macro.** Como as classes estao levemente desbalanceadas, a media macro da o mesmo peso para cada conceito. Tambem acompanhamos acuracia, acuracia balanceada, precisao macro e recall macro.

**Para o grupo:** quem for rodar os outros algoritmos deve usar os mesmos `SEED`, `cv` e `scoring` definidos abaixo, para que os resultados possam ser comparados.

In [ ]:
SEED = 6767

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEED)

scoring = {
    'f1_macro': 'f1_macro',
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'precision_macro': 'precision_macro',
    'recall_macro': 'recall_macro',
}

## 8. Definicao dos modelos

Usamos os hiperparametros padrao do scikit-learn, ja que a otimizacao fica para o T2. Os algoritmos tem vieses indutivos diferentes:

- **KNN**: baseado em instancias, nao parametrico. Classifica pela classe dos vizinhos mais proximos, entao depende de distancias e precisa de normalizacao.
- **Random Forest**: ensemble de arvores de decisao. Nao depende da escala dos atributos, entao nao usa normalizacao.
- **Rede Neural (MLP)**: modelo nao linear treinado por otimizacao (gradiente). Precisa de normalizacao para convergir bem. Aumentamos `max_iter` de 200 para 1000 apenas para o otimizador conseguir convergir (com 200 iteracoes ele para antes e gera `ConvergenceWarning`), isso nao eh ajuste de hiperparametro.
- **LightGBM**: ensemble de árvores por gradient boosting. Não depende da escala dos atributos, então não usa normalizacão. Usamos os padroes da biblioteca (100 arvores, `learning_rate=0.1`, até 31 folhas);
- **SVM (RBF)**: busca a fronteira de decisão com a maior margem possível entre as classes, resolvendo um problema de otimização convexa. Com o kernel RBF (padrao do scikit-learn), a fronteira é não linear no espaco original e o modelo é não paramétrico. Como o kernel depende de distâncias, precisa de normalização.
- **Naive Bayes (Gaussiano)**: modelo probabilistico, assume atributos independentes dado a classe e distribuicao normal de cada atributo. Nao depende da escala dos atributos, entao nao usa normalizacao.
- **Regressao Logistica**: modelo linear treinado por otimizacao. Usa normalizacao para o otimizador convergir bem e para a regularizacao tratar todos os atributos na mesma escala.

O `StandardScaler` fica dentro do `Pipeline`, entao eh ajustado apenas nos dados de treino de cada fold.

In [ ]:
models = {
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier()),
    ]),
    'Random Forest': RandomForestClassifier(random_state=SEED, n_jobs=1),
    'Rede Neural (MLP)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', MLPClassifier(max_iter=1000, random_state=SEED)),
    ]),
    'LightGBM': LGBMClassifier(random_state=SEED, verbose=-1, n_jobs=1),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(random_state=SEED)),
    ]),
    'Naive Bayes': GaussianNB(),
    'Regressao Logistica': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(random_state=SEED)),
    ]),
}

## 9. Execucao do spot-checking

Cada modelo eh treinado e avaliado nas 50 divisoes. Os resultados de cada execucao sao guardados em `resultados_spotcheck.csv`, para juntar com os resultados dos outros algoritmos do grupo.

In [ ]:
resultados = []

for nome, modelo in models.items():
    scores = cross_validate(modelo, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    print(f"{nome}: tempo médio de treino = {scores['fit_time'].mean():.2f}s por fold")

    for metrica in scoring:
        for execucao, valor in enumerate(scores[f'test_{metrica}']):
            resultados.append({'modelo': nome, 'execucao': execucao, 'metrica': metrica, 'valor': valor})

df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv('resultados_spotcheck.csv', index=False)
print(f"\nResultados salvos: {df_resultados.shape}")

## 10. Resultados

In [ ]:
# Média e desvio padrão de cada métrica por modelo, ordenado pelo F1 macro
resumo = df_resultados.groupby(['modelo', 'metrica'])['valor'].agg(['mean', 'std']).unstack('metrica')
resumo = resumo.sort_values(('mean', 'f1_macro'), ascending=False)

tabela = pd.DataFrame(index=resumo.index)
for metrica in scoring:
    tabela[metrica] = resumo[('mean', metrica)].map('{:.4f}'.format) + ' ± ' + resumo[('std', metrica)].map('{:.4f}'.format)

display(tabela)

In [ ]:
# Distribuição do F1 macro nas 50 execuções
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_resultados[df_resultados['metrica'] == 'f1_macro'], x='modelo', y='valor')
plt.title('F1 macro por modelo (5 folds x 10 repetições)')
plt.ylabel('F1 macro')
plt.xlabel('')
plt.show()

# Todas as métricas
g = sns.catplot(data=df_resultados, x='modelo', y='valor', col='metrica', kind='box', col_wrap=3, sharey=False, height=4)
g.set_xticklabels(rotation=20)
g.set_axis_labels('', 'valor')
plt.show()

In [ ]:
# Matrizes de confusão (uma validação cruzada de 5 folds), normalizadas por linha
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
conceitos = ['A', 'B', 'C', 'D']

fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5))
for ax, (nome, modelo) in zip(axes, models.items()):
    y_pred = cross_val_predict(modelo, X, y, cv=skf, n_jobs=-1)
    cm = confusion_matrix(y, y_pred, labels=conceitos, normalize='true')
    ConfusionMatrixDisplay(cm, display_labels=conceitos).plot(ax=ax, cmap='Blues', values_format='.2f', colorbar=False)
    ax.set_title(nome)
plt.tight_layout()
plt.show()

## 11. Discussao dos resultados

*(A ser escrita pelo grupo.)* Pontos a discutir:

- Qual modelo teve o melhor F1 macro, e se a diferenca para os outros eh grande em relacao ao desvio padrao.
- Qual modelo eh mais estavel (menor variabilidade nos boxplots).
- Custo de treino de cada modelo.
- Nas matrizes de confusao: os erros acontecem principalmente entre conceitos vizinhos (ex.: A e B)? Como os conceitos sao ordinais, errar A por B eh menos grave que A por D, e as metricas usadas nao levam isso em conta.
- Quais modelos seguem para o T2, junto com os resultados dos outros algoritmos do grupo.